In [1]:
import findspark
findspark.init()

In [2]:
# Steps to create a simple silver sales table for practice

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [4]:
spark = SparkSession.builder \
    .appName("SilverToGold Transformation") \
    .getOrCreate()

In [3]:


# --- 1. Define the Schema using StringType for the Date column ---
schema = StructType([
    StructField("Product",       StringType(),  True),
    StructField("OrderID",       IntegerType(), True),
    StructField("Date",          StringType(),  True),  # Note: Defined as STRING
    StructField("Customer",      StringType(),  True),
    StructField("Quantity",      IntegerType(), True),
    StructField("Price",         DoubleType(),  True),
    StructField("standard_name", StringType(),  True)
])

# --- 2. Define the Data ---
data = [
    ("Widget C", 1003, "2025-01-01", "Delta Inc",  1, 120.00, "Widget Gamma"),
    ("Widget B", 1002, "2025-01-01", "Beta LLC",   2,  40.00, "Widget Beta"),
    ("Widget B", 1004, "2025-01-01", "Acme Corp",  3,  40.00, "Widget Beta"),
    ("Widget A", 1005, "2025-01-02", "Acme Corp",  5,  25.00, "Widget Alpha"),
    ("Widget B", 1006, "2025-01-02", "Omega Corp", 3,  40.00, "Widget Beta"),
    ("Widget A", 1001, "2025-01-01", "Acme Corp",  5,  25.00, "Widget Alpha"),
    ("Widget D", 1007, "2025-01-01", "Omega Co",   3,  90.00, "Widget Delta"),
    ("Widget A", 1008, "2025-01-02", "Zeta Group", 4,  54.29, "Widget Alpha")
]

In [ ]:
# --- 3. Create the DataFrame ---
df_silver = spark.createDataFrame(data, schema=schema)

# --- 4. Fix the Data Type: Convert the String 'Date' to DateType ---
# The correct format is "yyyy-MM-dd" for the strings in the data variable.
df_silver = df_silver.withColumn(
    "Date",
    to_date(col("Date"), "yyyy-MM-dd")
)

print("--- Generated Silver Dataset ---")
df_silver.printSchema()
display(df_silver)

# --- 5. Save the DataFrame as a Delta Table ---
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("small_silver_sales")

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_monthly_sales_summary (
    product        STRING,
    sales_month    DATE,
    total_sales    DOUBLE,
    total_quantity DOUBLE,
    average_price  DOUBLE
) USING DELTA;

In [ ]:
%%sql
INSERT INTO gold_monthly_sales_summary
    SELECT
        Product                      AS product,
        DATE_TRUNC('month', Date)    AS sales_month,
        SUM(Price * Quantity)        AS total_sales,
        SUM(Quantity)                AS total_quantity,
        AVG(Price)                   AS average_price
    FROM silver_sales
    GROUP BY
        Product,
        DATE_TRUNC('month', Date);

In [ ]:
%%sql
ALTER TABLE gold_monthly_sales_summary ADD COLUMN net_sales DOUBLE;

UPDATE gold_monthly_sales_summary
SET net_sales = total_sales * 0.9;

SELECT *
FROM gold_monthly_sales_summary;

In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_monthly_sales_summary_partitioned
PARTITIONED BY (Product)
AS SELECT * FROM gold_monthly_sales_summary;

SELECT *
FROM gold_monthly_sales_summary_partitioned;

DESCRIBE gold_monthly_sales_summary_partitioned;

In [ ]:
%%sql
SELECT
    Product,
    SUM(net_sales) AS product_sales
FROM gold_monthly_sales_summary
WHERE sales_month >= '2025-01-01'
  AND sales_month <= '2025-03-31'
GROUP BY Product
ORDER BY product_sales DESC;